# GDELT

## Raw event feed — 15-minute snapshot (GDELT 2.0)

Moved out of `source-exploration.ipynb`. GDELT 2.0's 15-min export has 61 columns and no header row — the file has no way to tell you what's in it, so `V2_COLS` below is the schema, kept by position. `usecols` then only actually loads what we use.

In [1]:
import pandas as pd, requests, io, zipfile
from datetime import datetime, timedelta, timezone

# GDELT 2.0 event export schema, fixed column order (no header row in the file)
V2_COLS = (["GlobalEventID","SQLDATE","MonthYear","Year","FractionDate"]
  + [f"Actor{n}{f}" for n in (1,2) for f in
     ["Code","Name","CountryCode","KnownGroupCode","EthnicCode",
      "Religion1Code","Religion2Code","Type1Code","Type2Code","Type3Code"]]
  + ["IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
     "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone"]
  + [f"{p}Geo_{f}" for p in ("Actor1","Actor2","Action") for f in
     ["Type","FullName","CountryCode","ADM1Code","ADM2Code","Lat","Long","FeatureID"]]
  + ["DATEADDED","SOURCEURL"])

USECOLS = ["GlobalEventID","Actor1Name","Actor2Name","EventRootCode","NumMentions",
           "SOURCEURL","ActionGeo_CountryCode","GoldsteinScale","AvgTone"]

# lastupdate.txt can point at a file that isn't actually up yet — a real gap in
# GDELT's own publishing, not just us being early. Step back 15 min at a time
# until one actually exists, instead of trusting the pointer blindly.
t = datetime.strptime(requests.get("http://data.gdeltproject.org/gdeltv2/lastupdate.txt")
                       .text.split()[2].rsplit("/", 1)[-1][:14], "%Y%m%d%H%M%S")
for attempt in range(5):
    url = t.strftime("http://data.gdeltproject.org/gdeltv2/%Y%m%d%H%M00.export.CSV.zip")
    resp = requests.get(url)
    if resp.status_code == 200:
        break
    t -= timedelta(minutes=15)
else:
    raise RuntimeError("no recent 15-min file found")

z = zipfile.ZipFile(io.BytesIO(resp.content))
df = pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None, names=V2_COLS, usecols=USECOLS, low_memory=False)

# biggest story in this 15-min window, by mention count
df.nlargest(5, "NumMentions")[["Actor1Name","Actor2Name","EventRootCode","NumMentions","SOURCEURL"]]

,Actor1Name,Actor2Name,EventRootCode,NumMentions,SOURCEURL
968,RUSSIA,NaN,19,110,https://www.malverngazette.co.uk/news/national...
972,RUSSIA,EMPLOYEE,19,90,https://www.malverngazette.co.uk/news/national...
1104,AMERICAN,GREEK,2,80,https://www.newsshopper.co.uk/news/national/26...
150,NaN,UKRAINE,19,62,https://www.malverngazette.co.uk/news/national...
370,POLICE,NaN,5,50,https://www.brentwoodlive.co.uk/news/national/...


### Mood ranking — net cooperation/conflict by country

In [2]:
# roughest/calmest countries in this window, by mean Goldstein score
mood = (df.groupby("ActionGeo_CountryCode")
          .agg(events=("GlobalEventID","size"),
               goldstein=("GoldsteinScale","mean"),
               tone=("AvgTone","mean"))
          .query("events >= 7")
          .sort_values("goldstein"))

print(mood.head(5))   # roughest
print(mood.tail(5))   # calmest

                       events  goldstein      tone
ActionGeo_CountryCode                             
SO                         43  -3.360465 -6.750392
MY                         11  -2.818182 -3.116610
NI                        207  -2.282609 -2.854678
CE                         67  -1.967164 -3.747302
UP                         13  -1.507692 -5.945996
                       events  goldstein      tone
ActionGeo_CountryCode                             
MI                         45   1.771111 -0.873000
CA                          8   2.375000  1.788435
TX                         12   2.400000  2.873563
PE                         12   8.000000 -1.684920
BH                         12   8.000000 -1.684920


### Best/worst relationships

6-hour window (24 files) instead of one 15-min snapshot, so country pairs actually have enough events to rank.

In [3]:
from datetime import datetime, timedelta, timezone

PAIR_COLS = ["GlobalEventID","Actor1Name","Actor2Name","Actor1CountryCode",
             "Actor2CountryCode","GoldsteinScale","NumMentions","SOURCEURL"]

t = datetime.now(timezone.utc).replace(second=0, microsecond=0)
t -= timedelta(minutes=t.minute % 15 + 15)

frames = []
for i in range(24):
    url = (t - timedelta(minutes=15*i)).strftime(
        "http://data.gdeltproject.org/gdeltv2/%Y%m%d%H%M00.export.CSV.zip")
    try:
        z = zipfile.ZipFile(io.BytesIO(requests.get(url, timeout=30).content))
        frames.append(pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None,
                                   names=V2_COLS, usecols=PAIR_COLS, low_memory=False))
    except Exception:
        pass
recent = pd.concat(frames)

# only cross-border events, pair sorted so US<->CN and CN<->US collapse into one row
p = recent.dropna(subset=["Actor1CountryCode","Actor2CountryCode"])
p = p[p.Actor1CountryCode != p.Actor2CountryCode].copy()
p["pair"] = [" <-> ".join(sorted(x)) for x in zip(p.Actor1CountryCode, p.Actor2CountryCode)]

board = (p.groupby("pair")
          .agg(events=("GlobalEventID","size"), mood=("GoldsteinScale","mean"))
          .query("events >= 15")
          .sort_values("mood"))

print("WORST RELATIONSHIPS TODAY\n", board.head(10), "\n")
print("BEST RELATIONSHIPS TODAY\n", board.tail(10))

WORST RELATIONSHIPS TODAY
              events      mood
pair                         
RUS <-> UKR     165 -5.946667
PRK <-> RUS      16 -5.925000
FRA <-> RUS      15 -2.966667
JPN <-> PRK      15 -2.960000
IRN <-> ISR      26 -2.192308
CUB <-> USA      25 -1.800000
CAN <-> USA      38 -1.768421
UKR <-> USA      18 -1.611111
SAU <-> YEM      27 -1.440741
BRA <-> USA      56 -0.982143 

BEST RELATIONSHIPS TODAY
              events      mood
pair                         
GBR <-> IRL      17  2.311765
ISR <-> LBN      51  2.490196
GRC <-> USA      17  2.664706
EGY <-> ISR      19  2.789474
PAK <-> TUR      16  3.000000
IRN <-> OMN      67  3.240299
IRN <-> PAK      30  3.300000
ISR <-> ITA      22  3.972727
ITA <-> LBN      19  4.000000
GBR <-> USA      25  4.076000


In [4]:
# drill into the worst pair — what actually happened
worst = board.index[0]
story = p[p.pair == worst].nlargest(1, "NumMentions").iloc[0]
print(f"\n{worst}: {story.Actor1Name} -> {story.Actor2Name}")
print(story.SOURCEURL)


RUS <-> UKR: UKRAINIAN -> RUSSIA
http://www.strategypage.com/%5Chtmw%5Chtlog%5Carticles%5C2026080453242.aspx


## What is each Canadian city about?

30 days of GDELT 1.0 daily files (one request/day instead of 96) — a first look at whether cities have a distinct topical "fingerprint" relative to the national mix.

In [5]:
from datetime import date, timedelta

# GDELT 1.0 daily schema — no ADM2Code in the geo fields, otherwise same idea as V2_COLS
V1_COLS = (["GlobalEventID","SQLDATE","MonthYear","Year","FractionDate"]
  + [f"Actor{n}{f}" for n in (1,2) for f in
     ["Code","Name","CountryCode","KnownGroupCode","EthnicCode",
      "Religion1Code","Religion2Code","Type1Code","Type2Code","Type3Code"]]
  + ["IsRootEvent","EventCode","EventBaseCode","EventRootCode","QuadClass",
     "GoldsteinScale","NumMentions","NumSources","NumArticles","AvgTone"]
  + [f"{p}Geo_{f}" for p in ("Actor1","Actor2","Action") for f in
     ["Type","FullName","CountryCode","ADM1Code","Lat","Long","FeatureID"]]
  + ["DATEADDED","SOURCEURL"])

CITY_COLS = ["ActionGeo_CountryCode","ActionGeo_Type","ActionGeo_FullName","EventRootCode"]

frames = []
for i in range(1, 31):                      # last 30 days
    d = (date.today() - timedelta(days=i)).strftime("%Y%m%d")
    try:
        r = requests.get(f"http://data.gdeltproject.org/events/{d}.export.CSV.zip", timeout=60)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        day = pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None,
                           names=V1_COLS, usecols=CITY_COLS, low_memory=False)
        frames.append(day[(day.ActionGeo_CountryCode == "CA") &
                           (day.ActionGeo_Type == 4)])     # 4 = world city
    except Exception as e:
        print(d, "skipped", e)

In [6]:
ca = pd.concat(frames, ignore_index=True)

ca["city"] = ca.ActionGeo_FullName.str.split(",").str[0].str.strip()
ca["prov"] = ca.ActionGeo_FullName.str.split(",").str[1].str.strip()

CAMEO = {1:"Statement",2:"Appeal",3:"Intent to cooperate",4:"Consult",
         5:"Diplomatic coop",6:"Material coop",7:"Aid",8:"Yield",
         9:"Investigate",10:"Demand",11:"Disapprove",12:"Reject",
         13:"Threaten",14:"Protest",15:"Force posture",16:"Reduce relations",
         17:"Coerce",18:"Assault",19:"Fight",20:"Mass violence"}
ca["action"] = ca.EventRootCode.map(CAMEO)

# fingerprint: for each city, which action types are over-represented vs. the national mix
big = ca.city.value_counts().head(15).index
sub = ca[ca.city.isin(big)]

city_mix = pd.crosstab(sub.city, sub.action, normalize="index")
national = sub.action.value_counts(normalize=True)
lift = (city_mix / national).dropna(axis=1)

for c in big:
    top = lift.loc[c].nlargest(3)
    n = (sub.city == c).sum()
    print(f"{c:<12} n={n:<5} " + " | ".join(f"{k} {v:.1f}x" for k, v in top.items()))

Toronto      n=9390  Mass violence 2.9x | Fight 1.5x | Force posture 1.4x
Ottawa       n=7645  Demand 1.5x | Diplomatic coop 1.2x | Force posture 1.2x
Vancouver    n=4237  Protest 1.8x | Coerce 1.6x | Yield 1.4x
Quebec       n=3502  Reduce relations 1.4x | Intent to cooperate 1.3x | Material coop 1.3x
Montreal     n=2795  Assault 1.6x | Investigate 1.6x | Fight 1.4x
Calgary      n=2649  Investigate 1.8x | Protest 1.3x | Intent to cooperate 1.3x
Winnipeg     n=1983  Reduce relations 2.0x | Protest 1.4x | Statement 1.2x
Saskatchewan n=1391  Mass violence 3.4x | Appeal 1.3x | Aid 1.3x
Thunder Bay  n=770   Reduce relations 1.8x | Demand 1.5x | Fight 1.4x
Prince Edward Island n=578   Diplomatic coop 1.6x | Intent to cooperate 1.6x | Consult 1.3x
Sarnia       n=511   Material coop 1.6x | Appeal 1.5x | Demand 1.4x
Brockville   n=485   Protest 2.5x | Aid 1.6x | Diplomatic coop 1.5x
Sudbury      n=395   Force posture 3.7x | Material coop 1.3x | Statement 1.3x
Niagara Falls n=353   Appeal 1.6x |

## Country-day panel — fetch mechanics

Phase 2.2 target: a country-day tone/volume panel for ~5 countries over a few months. Countries: **CA, US, MX** (consistent with Wikipedia — same audience, already-known quirks) plus **UK** (the other "likely audience" country) and **GM/Germany** (the culturally-distinct pick from the Wikipedia country discussion, revisited here since GDELT makes it free — every daily file already contains every country, so widening from 3 to 5 costs zero extra requests, unlike Wikimedia's one-call-per-country).

**Gotcha caught while testing**: GDELT's `CountryCode` fields are **FIPS 10-4**, not ISO 3166-1 — `UK` and `GM`, not `GB`/`DE`. Using the ISO codes silently returned ~0 rows for Germany and near-nothing for "GB" (not a real match) instead of erroring, which would've been an easy miss. CA/US/MX happen to be identical in both standards, so this only bit the 2 new countries.

Before committing to a full backfill: confirm the daily file reliably covers all 5, and get a feel for how much daily volume each one actually has (GDELT's coverage is wire-service driven, so it's not evenly distributed — worth checking before trusting the panel).

In [74]:
COUNTRIES = ["CA", "US", "MX", "UK", "GM"]  # FIPS 10-4, not ISO — see note above

PANEL_COLS = ["ActionGeo_CountryCode", "GoldsteinScale", "AvgTone", "GlobalEventID"]

def fetch_gdelt_day(day: date, usecols: list[str] = V1_COLS, dtype: dict = None) -> pd.DataFrame:
    """One GDELT 1.0 daily file -> DataFrame with just the columns we asked for.

    dtype is needed for EventCode: it's zero-padded ("010", "0211") in the raw
    file, and pandas silently reads it as int (dropping the leading zero) unless
    told otherwise — a real bug we hit trying to look it up in the CAMEO table.
    """
    url = f"http://data.gdeltproject.org/events/{day.strftime('%Y%m%d')}.export.CSV.zip"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    return pd.read_csv(z.open(z.namelist()[0]), sep="\t", header=None,
                        names=V1_COLS, usecols=usecols, dtype=dtype, low_memory=False)


# last 5 days, all 5 countries at once — same file has everyone in it
for i in range(1, 6):
    d = date.today() - timedelta(days=i)
    day_df = fetch_gdelt_day(d, PANEL_COLS)
    counts = day_df[day_df.ActionGeo_CountryCode.isin(COUNTRIES)].groupby("ActionGeo_CountryCode").size()
    print(d, dict(counts))

2026-08-05 {'CA': np.int64(2676), 'GM': np.int64(759), 'MX': np.int64(852), 'UK': np.int64(5340), 'US': np.int64(36272)}
2026-08-04 {'CA': np.int64(2448), 'GM': np.int64(641), 'MX': np.int64(480), 'UK': np.int64(5696), 'US': np.int64(33823)}
2026-08-03 {'CA': np.int64(1355), 'GM': np.int64(497), 'MX': np.int64(491), 'UK': np.int64(5204), 'US': np.int64(27384)}
2026-08-02 {'CA': np.int64(1174), 'GM': np.int64(406), 'MX': np.int64(213), 'UK': np.int64(3213), 'US': np.int64(15316)}
2026-08-01 {'CA': np.int64(1537), 'GM': np.int64(508), 'MX': np.int64(360), 'UK': np.int64(3192), 'US': np.int64(21248)}


## Country-day tone/volume panel

The actual deliverable for this checklist item: one row per country per day — event count, mean Goldstein, mean tone — over the last ~3 months. One request per day (not per country), same as the mechanics check above.

In [8]:
PANEL_DAYS = 90  # ~3 months, "a few months" per the plan

rows = []
for i in range(1, PANEL_DAYS + 1):
    d = date.today() - timedelta(days=i)
    try:
        day_df = fetch_gdelt_day(d, PANEL_COLS)
    except Exception as e:
        print(d, "skipped", e)
        continue
    sub = day_df[day_df.ActionGeo_CountryCode.isin(COUNTRIES)]
    daily = (sub.groupby("ActionGeo_CountryCode")
                .agg(events=("GlobalEventID", "size"),
                     goldstein=("GoldsteinScale", "mean"),
                     tone=("AvgTone", "mean")))
    daily["date"] = d
    rows.append(daily.reset_index())

panel = pd.concat(rows, ignore_index=True).rename(columns={"ActionGeo_CountryCode": "country"})
panel = panel.sort_values(["country", "date"]).reset_index(drop=True)

print(f"{len(panel)} country-days across {panel.country.nunique()} countries, {panel.date.min()} to {panel.date.max()}")
panel.groupby("country")["events"].describe()[["mean", "min", "max"]]

450 country-days across 5 countries, 2026-05-07 to 2026-08-04


,mean,min,max
country,,,
CA,2594.344444,913.0,3746.0
GM,817.900000,352.0,1328.0
MX,487.777778,142.0,865.0
UK,5368.144444,3035.0,7479.0
US,29638.677778,12840.0,42807.0


## First derived observation — rolling baseline + deviation flags

The actual point of the panel: is a country's mood on a given day unusual *for that country*, not just in absolute terms — the "relative over raw" idea from the observation-spine model, made real.

Baseline for day D uses only the 14 days strictly before D (`shift(1)`) — otherwise today's own value would leak into its own baseline and dampen every deviation.

In [9]:
WINDOW = 14

by_country = panel.groupby("country")["tone"]
panel["tone_baseline"] = by_country.transform(lambda s: s.rolling(WINDOW, min_periods=WINDOW).mean().shift(1))
panel["tone_std"] = by_country.transform(lambda s: s.rolling(WINDOW, min_periods=WINDOW).std().shift(1))
panel["tone_z"] = (panel["tone"] - panel["tone_baseline"]) / panel["tone_std"]

# |z| > 1.5 -> today's mood is a real outlier relative to this country's own recent baseline
flagged = panel[panel.tone_z.abs() > 1.5].sort_values("tone_z")
flagged[flagged['country'] == "CA"][["country", "date", "tone", "tone_baseline", "tone_z"]]

,country,date,tone,tone_baseline,tone_z
47,CA,2026-06-23,-1.618046,-0.666715,-4.239348
87,CA,2026-08-02,-1.659977,-0.721359,-3.466769
66,CA,2026-07-12,-1.734284,-0.556615,-2.413802
72,CA,2026-07-18,-1.862968,-0.765603,-2.254437
31,CA,2026-06-07,-1.142186,-0.642990,-2.219604
52,CA,2026-06-28,-1.394689,-0.741948,-2.086733
36,CA,2026-06-12,-1.101961,-0.656421,-1.666888
46,CA,2026-06-22,-0.978576,-0.614561,-1.575151
22,CA,2026-05-29,-1.021273,-0.653666,-1.554644
74,CA,2026-07-20,-0.199536,-0.940508,1.514372


## GKG — Global Knowledge Graph (spike)

Separate table from Events, daily file confirmed at `gkg/<date>.gkg.csv.zip` (this is the older GKG 1.0 daily rollup, not the 15-min GKG 2.1 under `gdeltv2/`). Unlike Events, this file actually **has a header row** — no hardcoded column list needed. `THEMES` is the column to look at for the topic-classification question.

Just fetching it for now to poke around — not filtering/aggregating anything yet.

In [70]:
def fetch_gkg_day(day: date) -> pd.DataFrame:
    """One day of GDELT's GKG table. Has a real header, so no column list to maintain."""
    url = f"http://data.gdeltproject.org/gkg/{day.strftime('%Y%m%d')}.gkg.csv.zip"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    return pd.read_csv(z.open(z.namelist()[0]), sep="\t", low_memory=False)


gkg = fetch_gkg_day(date.today() - timedelta(days=2))

In [11]:
gkg.columns

Index(['DATE', 'NUMARTS', 'COUNTS', 'THEMES', 'LOCATIONS', 'PERSONS',
       'ORGANIZATIONS', 'TONE', 'CAMEOEVENTIDS', 'SOURCES', 'SOURCEURLS'],
      dtype='str')

In [12]:
gkg.head()

,DATE,NUMARTS,COUNTS,THEMES,LOCATIONS,PERSONS,ORGANIZATIONS,TONE,CAMEOEVENTIDS,SOURCES,SOURCEURLS
0,20260803,1,NaN,TAX_ETHNICITY;TAX_ETHNICITY_AUSTRIAN;CRISISLEX...,1#Austria#AU#AU#47.333333#13.333333#AU;1#Germa...,NaN,NaN,"-0.195121951219512,2.73170731707317,2.92682926...","1316578072,1316578073,1316578161,1316578180",thelocal.at,https://www.thelocal.at/20260803/can-foreign-r...
1,20260803,1,NaN,NaN,"2#Ohio, United States#US#USOH#40.3736#-82.7755...",bryce young;dave canales;robert hunt;kenny pic...,NaN,"0.769230769230769,2.69230769230769,1.923076923...",NaN,pittsburghstar.com,http://www.pittsburghstar.com/news/279221296/p...
2,20260803,1,KILL#3#animals#1#United Kingdom#UK#UK#54#-4#UK...,CRISISLEX_T01_CAUTION_ADVICE;KILL;CRISISLEX_T0...,1#United Kingdom#UK#UK#54#-4#UK,NaN,NaN,"-1.00806451612903,2.01612903225806,3.024193548...",NaN,aol.co.uk,https://www.aol.co.uk/articles/uk-zoo-suddenly...
3,20260803,1,"EVACUATION#60##2#Washington, United States#US#...",TAX_ETHNICITY;TAX_ETHNICITY_INDIAN;NATURAL_DIS...,"2#Oregon, United States#US#USOR#44.5672#-122.1...",young kwak allen sather;chris jordan;heather r...,national weather service;u s veterans affairs;...,"-3.96744659206511,0.610376398779247,4.57782299...","1316521943,1316521947,1316521978,1316521980,13...",butlereagle.com,https://www.butlereagle.com/20260802/eastern-w...
4,20260803,1,NaN,TAX_ECON_PRICE;TAX_FNCACT;TAX_FNCACT_RETAILER;...,1#Hong Kong#HK#HK#22.25#114.166667#HK,NaN,hong kong stock exchange;reuters,"-1.23456790123457,0.925925925925926,2.16049382...",NaN,dealstreetasia.com,https://www.dealstreetasia.com/stories/shein-i...


In [13]:
themes = gkg['THEMES']

In [14]:
theme_lists = list(themes.str.split(";"))

In [15]:
unique_items = set()
for item in theme_lists:
    if type(item) is float:
        continue
        
    for theme in item:
        if theme:
            unique_items.add(theme)

In [16]:
sorted(list(unique_items))

['ACT_FORCEPOSTURE',
 'ACT_HARMTHREATEN',
 'ACT_MAKESTATEMENT',
 'ACT_YIELD',
 'AFFECT',
 'AGRICULTURE',
 'AID_ECONOMIC',
 'AID_HUMANITARIAN',
 'ALLIANCE',
 'APPOINTMENT',
 'ARMEDCONFLICT',
 'ARREST',
 'ASSASSINATION',
 'AUSTERITY',
 'AVIATION_INCIDENT',
 'BAN',
 'BLACK_MARKET',
 'BLOCKADE',
 'BORDER',
 'BULLYING',
 'CEASEFIRE',
 'CHARASMATIC_LEADERSHIP',
 'CHECKPOINT',
 'CLAIM_CREDIT',
 'CLOSURE',
 'CONFISCATION',
 'CONSTITUTIONAL',
 'CORRUPTION',
 'CRIME_CARTELS',
 'CRIME_COMMON_ROBBERY',
 'CRIME_ILLEGAL_DRUGS',
 'CRIME_LOOTING',
 'CRIME_STEAL_LIVESTOCK',
 'CRISISLEX_C01_CHILDREN_AND_EDUCATION',
 'CRISISLEX_C02_NEEDSPROVIDE_FOOD',
 'CRISISLEX_C03_WELLBEING_HEALTH',
 'CRISISLEX_C04_LOGISTICS_TRANSPORT',
 'CRISISLEX_C05_NEED_OF_SHELTERS',
 'CRISISLEX_C06_WATER_SANITATION',
 'CRISISLEX_C07_SAFETY',
 'CRISISLEX_C08_TELECOM',
 'CRISISLEX_CRISISLEXREC',
 'CRISISLEX_O01_WEATHER',
 'CRISISLEX_O02_RESPONSEAGENCIESATCRISIS',
 'CRISISLEX_T01_CAUTION_ADVICE',
 'CRISISLEX_T02_INJURED',
 'CRISISLE

### Theme-enriched events table

Join Events to GKG via `CAMEOEVENTIDS`, so each event gets the theme tags of the article(s) it came from.

**First attempt fanned out**: a plain explode-and-merge turned 91,182 events into 227,238 rows. Cause: popular stories get covered by dozens of syndicated outlets, and *every one* of those articles' GKG rows references the same `GlobalEventID` (found one wildfire event referenced by 24 different outlets). So the join is many-articles-to-one-event, not one-to-one — a naive merge duplicates the event once per covering article.

**Fix**: pool all themes mentioned across every article that covers an event into one set, so there's still exactly one output row per event.

In [47]:
GKG_DAY = date.today() - timedelta(days=2)   # same day as `gkg` above

EVENT_COLS = ["GlobalEventID", "EventRootCode", "GoldsteinScale", "AvgTone",
              "ActionGeo_CountryCode", "SOURCEURL"]
events = fetch_gdelt_day(GKG_DAY, EVENT_COLS)

# one row per (event id, covering article) — this is where the fan-out happens
pairs = gkg[["CAMEOEVENTIDS", "THEMES"]].dropna(subset=["CAMEOEVENTIDS", "THEMES"]).copy()
pairs["GlobalEventID"] = pairs["CAMEOEVENTIDS"].str.split(",")
pairs = pairs.explode("GlobalEventID")
pairs["GlobalEventID"] = pairs["GlobalEventID"].astype("int64")

# collapse back down to one row per event: union of every covering article's theme tags
theme_by_event = (
    pairs.groupby("GlobalEventID")["THEMES"]
    .apply(lambda s: sorted(set(";".join(s).split(";")) - {""}))
)

enriched = events.merge(theme_by_event.rename("themes"), on="GlobalEventID", how="left")

print(f"{len(enriched)} rows (events had {len(events)} — should match exactly)")
print(f"{enriched['themes'].notna().mean():.1%} of events got themes")
enriched.dropna(subset=["themes"]).head()

91182 rows (events had 91182 — should match exactly)
96.8% of events got themes


,GlobalEventID,EventRootCode,GoldsteinScale,AvgTone,ActionGeo_CountryCode,SOURCEURL,themes
0,1316520961,11,-2.0,-7.544141,US,https://attackofthefanboy.com/politics/lapd-of...,"[ARMEDCONFLICT, BAN, CRISISLEX_C07_SAFETY, CRI..."
1,1316520962,4,1.9,0.160622,AS,https://www.merimbulanewsweekly.com.au/story/9...,"[CRISISLEX_C04_LOGISTICS_TRANSPORT, CRISISLEX_..."
2,1316520963,4,1.9,0.160622,AS,https://www.merimbulanewsweekly.com.au/story/9...,"[CRISISLEX_C04_LOGISTICS_TRANSPORT, CRISISLEX_..."
3,1316520964,13,-4.4,-5.797101,AS,https://www.theguardian.com/australia-news/202...,"[MANMADE_DISASTER_IMPLIED, TAX_FNCACT, TAX_FNC..."
4,1316520965,1,0.0,-1.340483,AS,https://www.newcastleherald.com.au/story/93220...,"[CRISISLEX_C04_LOGISTICS_TRANSPORT, CRISISLEX_..."


### Decoding event codes — the actual CAMEO verb, not just the root bucket

We've only been decoding `EventRootCode` (20 broad buckets, e.g. "Consult"). GDELT publishes the full CAMEO hierarchy — root (2-digit) → base (3-digit) → sub-code (4-digit, where one exists) — as a public lookup file. Confirmed it's real and its codes match our `EventCode` column's format exactly (zero-padded 3-4 digit strings) before trusting it.

This also answers the "is there a better way using Events alone" question from before: `Actor1Name`/`Actor2Name` are already the actors CAMEO assigned *to that specific event* — unlike GKG's `PERSONS`/`ORGANIZATIONS`, which have the same article-wide, not-event-scoped problem as `THEMES`. So the gloss below uses only Events' own columns, no GKG needed, and each row is precisely one event's actors — no pooling, no fan-out.

In [72]:
cameo_codes = pd.read_csv(
    "https://www.gdeltproject.org/data/lookups/CAMEO.eventcodes.txt", sep="\t", dtype=str
).set_index("CAMEOEVENTCODE")["EVENTDESCRIPTION"].to_dict()

def decode_event_code(code: str) -> str:
    """Most specific description available — full code, else its 3-digit base, else 2-digit root."""
    return cameo_codes.get(code) or cameo_codes.get(code[:3]) or cameo_codes.get(code[:2], "unknown")

In [75]:
GLOSS_COLS = ["GlobalEventID", "Actor1Name", "Actor2Name", "EventCode",
              "GoldsteinScale", "ActionGeo_CountryCode", "SOURCEURL"]
glossed = fetch_gdelt_day(GKG_DAY, GLOSS_COLS, dtype={"EventCode": str})

glossed["verb"] = glossed["EventCode"].apply(decode_event_code)
glossed["gloss"] = (glossed["Actor1Name"].fillna("someone") + " "
                     + glossed["verb"].str.lower() + " "
                     + glossed["Actor2Name"].fillna("someone"))

glossed[["gloss", "GoldsteinScale", "SOURCEURL"]].sample(8, random_state=1)

,gloss,GoldsteinScale,SOURCEURL
74809,FARMER make pessimistic comment someone,-0.4,https://www.newsbug.info/rensselaer_republican...
72095,UKRAINE praise or endorse WASHINGTON,3.4,https://en.interfax.com.ua/news/general/119038...
8590,"KURDISTAN engage in material cooperation, not ...",6.0,https://www.dailybreeze.com/2026/08/02/fbi-rai...
16512,"someone use conventional military force, not s...",-10.0,https://www.yakimaherald.com/news/local/highwa...
74229,NIGERIA host a visit SCHOOL,2.8,https://www.premiumtimesng.com/opinion/900446-...
8639,ISRAELI make a visit ARMENIAN,1.9,https://massispost.com/2026/08/israels-exploit...
9842,"BUREAUCRAT yield, not specified below UNITED ...",5.0,https://www.zerohedge.com/political/youre-parr...
64732,"SAUDI appeal, not specified below QATAR",3.0,http://www.milwaukeesun.com/news/279221039/tru...


### Read-friendly headlines from the source URL

The CAMEO gloss above is too generic to be useful on its own ("BANK make statement, not specified below SENATE" tells you nothing real). Better source: most news sites put the actual headline in the URL path, and every event row already has `SOURCEURL` — no new fetch needed.

Design: keep one row per event (don't dedupe/aggregate here — `NumMentions` is only meaningful per row, and an article that spawned several events legitimately produces several rows). `headline` is just another enrichment column, same pattern as `verb`/`themes`. Any "top stories for city X" view is a query on top of this (group by city, sort by `NumMentions`), not something baked in here.

Cleaning applied, each caught on real slugs from this data: strip file extensions (`.htm`/`.html`), strip trailing numeric/hex ID tokens (`b3026806`, `101785840419026`, `9.7291077`), strip a leading CMS-style numeric prefix (`26437230.murderer-...`), and drop the result entirely if fewer than 3 real words survive (an ID-only slug like `article-16fa-50bf-...` isn't a headline, it's noise — better to have no headline than a fragment).

In [80]:
import re

_LEADING_ID_RE = re.compile(r"^\d+\.")            # CMS-style numeric prefix, e.g. "26437230.murderer-..."
_EXT_RE = re.compile(r"\.(html?|aspx?|php)$", re.IGNORECASE)
_ID_TOKEN_RE = re.compile(r"^(?=.*\d)[0-9a-f]+$", re.IGNORECASE)  # hex chars w/ >=1 digit -> an id, not a word
_NUMERIC_RE = re.compile(r"^[\d.]+$")                             # pure numbers/decimals -> ids, not words

def slug_to_headline(url: str) -> str | None:
    """Best-effort real headline straight from the URL — no article fetch needed."""
    segments = re.split(r"[/?]", url)
    candidates = [s for s in segments if s.count("-") >= 3]
    if not candidates:
        return None
    best = _EXT_RE.sub("", max(candidates, key=len))
    best = _LEADING_ID_RE.sub("", best)
    words = [w for w in best.replace("_", "-").split("-")
             if w and not _ID_TOKEN_RE.match(w) and not _NUMERIC_RE.match(w)]
    if len(words) < 3:
        return None
    text = " ".join(words)
    return text[0].upper() + text[1:]


HEADLINE_COLS = ["GlobalEventID", "ActionGeo_CountryCode", "ActionGeo_Type",
                  "ActionGeo_FullName", "SOURCEURL", "NumMentions"]
headlined = fetch_gdelt_day(GKG_DAY, HEADLINE_COLS)
headlined["headline"] = headlined["SOURCEURL"].apply(slug_to_headline)

print(f"{headlined['headline'].notna().mean():.1%} of events got a headline")

88.4% of events got a headline


In [84]:
headlined.dropna(subset=["headline"])[["headline", "NumMentions"]].sample(8)

,headline,NumMentions
88737,Malawi opposition parties demand namiwas immed...,2
40639,Washington badger mountain solar project cance...,2
16840,Number of people missing unknown after indones...,112
69697,South sudan political parties commit to peacef...,2
14968,Seven turbulent months two greater,10
66413,California scientists build surveillance robot...,2
87686,Alabama enforces taylors law delaying drivers ...,7
69184,Big rise suspensions police scotland lothians ...,1


## Top stories for a city, over a week

Puts the last two pieces together: fetch a week of city-level events for the countries we track, then query it for one city's top stories by mention count. Fetching and printing are separate cells, as requested — the fetch cell builds `week_events` once; the print cell is just a query on top of it, so you can re-run it for a different city without re-fetching.

In [86]:
WEEK_DAYS = 7
WEEK_COLS = ["GlobalEventID", "ActionGeo_CountryCode", "ActionGeo_Type",
             "ActionGeo_FullName", "SOURCEURL", "NumMentions"]

frames = []
for i in range(1, WEEK_DAYS + 1):
    d = date.today() - timedelta(days=i)
    try:
        day_df = fetch_gdelt_day(d, WEEK_COLS)
    except Exception as e:
        print(d, "skipped", e)
        continue
    day_df = day_df[(day_df.ActionGeo_CountryCode.isin(COUNTRIES))
                     & (day_df.ActionGeo_Type == 4)].copy()   # world city granularity only
    day_df["date"] = d
    frames.append(day_df)

week_events = pd.concat(frames, ignore_index=True)
week_events["city"] = week_events.ActionGeo_FullName.str.split(",").str[0].str.strip()
week_events["headline"] = week_events["SOURCEURL"].apply(slug_to_headline)

print(f"{len(week_events)} city-level events across {len(frames)} days")

40113 city-level events across 7 days


In [87]:
def top_stories(city: str, country: str = None, n: int = 10) -> pd.DataFrame:
    """Top n stories for a city this week, ranked by NumMentions.

    Dedupes by SOURCEURL, not by headline text — two different outlets
    covering the same story with a similar-looking slug are still two
    real, separately-mentioned stories, not one.
    """
    sub = week_events[week_events.city == city]
    if country:
        sub = sub[sub.ActionGeo_CountryCode == country]
    sub = sub.dropna(subset=["headline"]).drop_duplicates(subset="SOURCEURL")
    return sub.nlargest(n, "NumMentions")[["date", "headline", "NumMentions"]]


top_stories("Toronto", "CA")

,date,headline,NumMentions
7535,2026-08-05,Kleen hy dro gen inc is pleased to announce du...,240
7406,2026-08-05,Ontario teacher contract talks have stalled is...,220
2572,2026-08-06,Siu sexual assault complaint data,198
7562,2026-08-05,Weird nightmare books toronto shows,186
2,2026-08-06,Parking day toronto returns next month,152
29956,2026-08-01,Fountain asset corp announces appointment of n...,146
7303,2026-08-05,Ontario teacher contract talks have stalled is...,138
5827,2026-08-06,Toronto police arrest in july shooting at us c...,136
4066,2026-08-06,Toronto police to give update on july us consu...,123
103,2026-08-06,Government of canada supports the return of th...,109


In [91]:
van_stories = top_stories("Vancouver", "CA").to_dict(orient='records')
for item in van_stories:
    print(item['headline'], f'------ > {item["NumMentions"]}')

Fire country star bids tearful farewell to los angeles cbs diane farr ------ > 620
Dont skip these beach day essentials ------ > 192
Anthony gismondi top white wine picks ------ > 191
Fabulous fashion finds from bc designers ------ > 183
We need help letter petition circulates calling on eby to declare state of emergency due to cariboo fires ------ > 156
Sailing waits kick off busiest long weekend of the year for bc ferries ------ > 150
Flight attendants at canadas westjet go on strike in dispute over pay ------ > 125
Vancouver restaurant burdock co dinner ------ > 122
Global education communities corp announces executive appointments and leadership updates ------ > 111
We need help letter petition circulates calling on eby to declare state of emergency due to cariboo fires ------ > 100


In [79]:
list(glossed['gloss'].sample(8))

['BANK make statement, not specified below SENATE',
 'PROSECUTOR consult, not specified below MILWAUKEE',
 'PALESTINIAN provide aid, not specified below ISRAELI',
 'GOVERNMENT host a visit STUDENT',
 'COMMUNITY consult, not specified below someone',
 'GOVERNOR consult, not specified below UNITED STATES',
 'IRAN host a visit CITIZEN',
 'UNITED KINGDOM host a visit MIGRANT']

## Random exploration

In [57]:
events = fetch_gdelt_day(GKG_DAY)

In [58]:
events.head()

,GlobalEventID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,Actor2Geo_FeatureID,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1316520961,20250803,202508,2025,2025.5836,NaN,NaN,NaN,NaN,NaN,...,1662328,3,"Los Angeles, California, United States",US,USCA,34.0522,-118.244,1662328,20260803,https://attackofthefanboy.com/politics/lapd-of...
1,1316520962,20250803,202508,2025,2025.5836,AUS,AUSTRALIA,AUS,NaN,NaN,...,NaN,4,"Gove Peninsula, Northern Territory, Australia",AS,AS03,-12.3333,136.817,-1576165,20260803,https://www.merimbulanewsweekly.com.au/story/9...
2,1316520963,20250803,202508,2025,2025.5836,AUS,AUSTRALIA,AUS,NaN,NaN,...,NaN,4,"Nhulunbuy, Northern Territory, Australia",AS,AS03,-12.1865,136.782,-1591473,20260803,https://www.merimbulanewsweekly.com.au/story/9...
3,1316520964,20250803,202508,2025,2025.5836,JUD,ADVOCATE,NaN,NaN,NaN,...,-1561728,4,"Brisbane, Queensland, Australia",AS,AS04,-27.5000,153.017,-1561728,20260803,https://www.theguardian.com/australia-news/202...
4,1316520965,20260727,202607,2026,2026.5671,AUSGOV,AUSTRALIA,AUS,NaN,NaN,...,AS,1,Australia,AS,AS,-25.0000,135.000,AS,20260803,https://www.newcastleherald.com.au/story/93220...


In [62]:
events.columns

Index(['GlobalEventID', 'SQLDATE', 'MonthYear', 'Year', 'FractionDate',
       'Actor1Code', 'Actor1Name', 'Actor1CountryCode', 'Actor1KnownGroupCode',
       'Actor1EthnicCode', 'Actor1Religion1Code', 'Actor1Religion2Code',
       'Actor1Type1Code', 'Actor1Type2Code', 'Actor1Type3Code', 'Actor2Code',
       'Actor2Name', 'Actor2CountryCode', 'Actor2KnownGroupCode',
       'Actor2EthnicCode', 'Actor2Religion1Code', 'Actor2Religion2Code',
       'Actor2Type1Code', 'Actor2Type2Code', 'Actor2Type3Code', 'IsRootEvent',
       'EventCode', 'EventBaseCode', 'EventRootCode', 'QuadClass',
       'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone',
       'Actor1Geo_Type', 'Actor1Geo_FullName', 'Actor1Geo_CountryCode',
       'Actor1Geo_ADM1Code', 'Actor1Geo_Lat', 'Actor1Geo_Long',
       'Actor1Geo_FeatureID', 'Actor2Geo_Type', 'Actor2Geo_FullName',
       'Actor2Geo_CountryCode', 'Actor2Geo_ADM1Code', 'Actor2Geo_Lat',
       'Actor2Geo_Long', 'Actor2Geo_FeatureID', 'Action

In [66]:
events['Actor2Name'].value_counts()[20:50]

Actor2Name
AUSTRALIA         504
SPAIN             476
OMAN              374
THE US            365
HOSPITAL          364
WASHINGTON        355
FRANCE            351
RESIDENTS         344
ISRAELI           326
CANADA            324
AMERICAN          323
UNIVERSITY        309
ADMINISTRATION    297
AUTHORITIES       294
MOROCCO           291
CONGRESS          284
WORKER            268
MIGRANT           264
RUSSIAN           260
SAUDI ARABIA      259
IRANIAN           257
CRIMINAL          248
VOTER             248
INDUSTRY          246
EMPLOYEE          245
IRAQ              240
PALESTINIAN       234
IRELAND           230
PRISON            228
JAPAN             220
Name: count, dtype: int64

In [55]:
gkg.head()

,DATE,NUMARTS,COUNTS,THEMES,LOCATIONS,PERSONS,ORGANIZATIONS,TONE,CAMEOEVENTIDS,SOURCES,SOURCEURLS
0,20260803,1,NaN,TAX_ETHNICITY;TAX_ETHNICITY_AUSTRIAN;CRISISLEX...,1#Austria#AU#AU#47.333333#13.333333#AU;1#Germa...,NaN,NaN,"-0.195121951219512,2.73170731707317,2.92682926...","1316578072,1316578073,1316578161,1316578180",thelocal.at,https://www.thelocal.at/20260803/can-foreign-r...
1,20260803,1,NaN,NaN,"2#Ohio, United States#US#USOH#40.3736#-82.7755...",bryce young;dave canales;robert hunt;kenny pic...,NaN,"0.769230769230769,2.69230769230769,1.923076923...",NaN,pittsburghstar.com,http://www.pittsburghstar.com/news/279221296/p...
2,20260803,1,KILL#3#animals#1#United Kingdom#UK#UK#54#-4#UK...,CRISISLEX_T01_CAUTION_ADVICE;KILL;CRISISLEX_T0...,1#United Kingdom#UK#UK#54#-4#UK,NaN,NaN,"-1.00806451612903,2.01612903225806,3.024193548...",NaN,aol.co.uk,https://www.aol.co.uk/articles/uk-zoo-suddenly...
3,20260803,1,"EVACUATION#60##2#Washington, United States#US#...",TAX_ETHNICITY;TAX_ETHNICITY_INDIAN;NATURAL_DIS...,"2#Oregon, United States#US#USOR#44.5672#-122.1...",young kwak allen sather;chris jordan;heather r...,national weather service;u s veterans affairs;...,"-3.96744659206511,0.610376398779247,4.57782299...","1316521943,1316521947,1316521978,1316521980,13...",butlereagle.com,https://www.butlereagle.com/20260802/eastern-w...
4,20260803,1,NaN,TAX_ECON_PRICE;TAX_FNCACT;TAX_FNCACT_RETAILER;...,1#Hong Kong#HK#HK#22.25#114.166667#HK,NaN,hong kong stock exchange;reuters,"-1.23456790123457,0.925925925925926,2.16049382...",NaN,dealstreetasia.com,https://www.dealstreetasia.com/stories/shein-i...


In [56]:
df.head()

,GlobalEventID,Actor1Name,Actor2Name,EventRootCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_CountryCode,SOURCEURL
0,1316863431,UNITED STATES,NaN,2,3.0,2,-2.915452,US,https://www.kmxt.org/news/2026-08-04/wildflowe...
1,1316863432,AUSSIE,AUSTRALIA,16,-7.0,4,-4.659498,AS,https://www.dailymail.com/news/article-1602949...
2,1316863433,AUSSIE,NaN,16,-7.0,6,-4.659498,AS,https://www.dailymail.com/news/article-1602949...
3,1316863434,AUSTRALIA,AUSTRALIA,19,-10.0,4,-4.659498,AS,https://www.dailymail.com/news/article-1602949...
4,1316863435,STUDENT,AUSTRALIA,14,-6.5,5,-4.659498,AS,https://www.dailymail.com/news/article-1602949...


In [49]:
ds = enriched.to_dict(orient='records')

In [53]:
ds[17]

{'GlobalEventID': 1316520978,
 'EventRootCode': 4,
 'GoldsteinScale': 1.9,
 'AvgTone': 0.91221443375822,
 'ActionGeo_CountryCode': 'AS',
 'SOURCEURL': 'https://www.begadistrictnews.com.au/story/9322361/river-clean-up-flows-into-data-centre-climate-debate/',
 'themes': ['AFFECT',
  'AGRICULTURE',
  'ALLIANCE',
  'APPOINTMENT',
  'CRISISLEX_C02_NEEDSPROVIDE_FOOD',
  'CRISISLEX_C04_LOGISTICS_TRANSPORT',
  'CRISISLEX_C06_WATER_SANITATION',
  'CRISISLEX_C07_SAFETY',
  'CRISISLEX_CRISISLEXREC',
  'CRISISLEX_T01_CAUTION_ADVICE',
  'CRISISLEX_T04_INFRASTRUCTURE',
  'CRISISLEX_T11_UPDATESSYMPATHY',
  'CURFEW',
  'DELAY',
  'ECON_HOUSING_PRICES',
  'ECON_STOCKMARKET',
  'ECON_TAXATION',
  'ECON_WORLDCURRENCIES',
  'ECON_WORLDCURRENCIES_DOLLAR',
  'EDUCATION',
  'ELECTION',
  'ENV_GREEN',
  'ENV_OIL',
  'ENV_WATERWAYS',
  'EPU_CATS_FISCAL_POLICY',
  'EPU_CATS_MIGRATION_FEAR_FEAR',
  'EPU_CATS_MIGRATION_FEAR_MIGRATION',
  'EPU_CATS_TAXES',
  'EPU_ECONOMY',
  'EPU_ECONOMY_HISTORIC',
  'EPU_POLICY',